Phase 3 – Demand Signal

Imports and Project Paths

In [29]:
# ============================================================
# PHASE 3 — COVERAGE ANALYSIS
# CELL 1: LOAD PROCESSED DATASETS
# ============================================================

from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent

PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"
OUTPUT_DATA = PROJECT_ROOT / "data" / "outputs"

OUTPUT_DATA.mkdir(parents=True, exist_ok=True)

print("Notebook:", NOTEBOOK_DIR)
print("Project:", PROJECT_ROOT)
print("Processed Exists:", PROCESSED_DATA.exists())
print("Output Exists:", OUTPUT_DATA.exists())

Notebook: d:\sivahitesh\Desktop\Tail_project\Notebooks
Project: d:\sivahitesh\Desktop\Tail_project
Processed Exists: True
Output Exists: True


In [30]:
# ============================================================
# CELL 2 — PANEL COVERAGE KEY & STRUCTURE VALIDATION
# ============================================================

print("=== PANEL COVERAGE STRUCTURE ===")

print("Columns:")
print(panel_coverage.columns.tolist())

print("\nDistrict ID columns:")
print([
    col for col in panel_coverage.columns
    if "district" in col.lower()
])

print("\nUnique normalized district IDs:")
print(
    panel_coverage["District_Ref_ID_normalized"].nunique()
)

print("\nUnique months:")
print(
    panel_coverage["Month"].nunique()
)

print("\nUnique District × Month keys:")
print(
    panel_coverage[
        ["District_Ref_ID_normalized", "Month"]
    ]
    .drop_duplicates()
    .shape[0]
)

print("\nMissing normalized district IDs:")
print(
    panel_coverage[
        "District_Ref_ID_normalized"
    ].isna().sum()
)

=== PANEL COVERAGE STRUCTURE ===
Columns:
['District_Ref_ID', 'Month', 'District_Name', 'State', 'Sampled_Outlets', 'Estimated_Total_Outlets', 'Panel_Coverage_Pct', 'Audit_Reliability_Tier', 'District_Ref_ID_normalized']

District ID columns:
['District_Ref_ID', 'District_Name', 'District_Ref_ID_normalized']

Unique normalized district IDs:
340

Unique months:
24

Unique District × Month keys:
8160

Missing normalized district IDs:
0


In [31]:
# ============================================================
# CELL 3 — DISTRICT × MONTH KEY UNIQUENESS
# ============================================================

print("=== DISTRICT × MONTH KEY VALIDATION ===")

duplicate_mask = panel_coverage.duplicated(
    subset=[
        "District_Ref_ID_normalized",
        "Month"
    ],
    keep=False
)

duplicate_rows = panel_coverage[duplicate_mask]

print(
    "Duplicate District × Month rows:",
    len(duplicate_rows)
)

print(
    "Unique duplicated District × Month keys:",
    duplicate_rows[
        [
            "District_Ref_ID_normalized",
            "Month"
        ]
    ].drop_duplicates().shape[0]
)

if len(duplicate_rows) == 0:
    print("\n✓ One record exists for every District × Month key.")
else:
    print("\n⚠ Duplicate District × Month keys found.")
    display(
        duplicate_rows[
            [
                "District_Ref_ID",
                "District_Ref_ID_normalized",
                "Month"
            ]
        ].head(20)
    )

=== DISTRICT × MONTH KEY VALIDATION ===
Duplicate District × Month rows: 0
Unique duplicated District × Month keys: 0

✓ One record exists for every District × Month key.


In [32]:
# ============================================================
# CELL 4 — 24-MONTH MEAN PANEL COVERAGE
# ============================================================

coverage_summary = (
    panel_coverage
    .groupby(
        "District_Ref_ID_normalized",
        as_index=False
    )
    .agg(
        Mean_Panel_Coverage_Pct=(
            "Panel_Coverage_Pct",
            "mean"
        )
    )
)

print("=== 24-MONTH MEAN PANEL COVERAGE ===")

print(
    "Unique districts:",
    coverage_summary[
        "District_Ref_ID_normalized"
    ].nunique()
)

print(
    "Coverage summary rows:",
    len(coverage_summary)
)

display(
    coverage_summary.head(20)
)

=== 24-MONTH MEAN PANEL COVERAGE ===
Unique districts: 340
Coverage summary rows: 340


,District_Ref_ID_normalized,Mean_Panel_Coverage_Pct
0,DST_0001,85.037917
1,DST_0002,82.070417
2,DST_0003,83.369167
3,DST_0004,41.783333
4,DST_0005,15.612500
5,DST_0006,55.080833
6,DST_0007,72.070000
7,DST_0008,57.322500
8,DST_0009,68.176250
9,DST_0010,36.100000


In [33]:
# ============================================================
# CELL 5 — MERGE COVERAGE WITH DISTRICT MASTER
# ============================================================

district_coverage = district_master.merge(
    coverage_summary,
    left_on="District_Code",
    right_on="District_Ref_ID_normalized",
    how="left",
    validate="one_to_one"
)

print("=== COVERAGE ↔ DISTRICT MASTER JOIN ===")

print(
    "District Master rows:",
    len(district_master)
)

print(
    "District Coverage rows:",
    len(district_coverage)
)

print(
    "Missing coverage values:",
    district_coverage[
        "Mean_Panel_Coverage_Pct"
    ].isna().sum()
)

print(
    "Unique District Codes:",
    district_coverage[
        "District_Code"
    ].nunique()
)

display(
    district_coverage[
        [
            "District_Code",
            "District_Name",
            "Mean_Panel_Coverage_Pct"
        ]
    ].head(20)
)

=== COVERAGE ↔ DISTRICT MASTER JOIN ===
District Master rows: 340
District Coverage rows: 340
Missing coverage values: 0
Unique District Codes: 340


,District_Code,District_Name,Mean_Panel_Coverage_Pct
0,DST_0001,Mumbai Suburban,85.037917
1,DST_0002,Mumbai City,82.070417
2,DST_0003,Pune,83.369167
3,DST_0004,Thane,41.783333
4,DST_0005,Palghar,15.612500
5,DST_0006,Nagpur,55.080833
6,DST_0007,Nashik,72.070000
7,DST_0008,Aurangabad,57.322500
8,DST_0009,Kolhapur,68.176250
9,DST_0010,Solapur,36.100000


In [34]:
# ============================================================
# CELL 6 — UNKNOWN MARKET CLASSIFICATION
# ============================================================

district_coverage["Coverage_Status"] = np.where(
    district_coverage["Mean_Panel_Coverage_Pct"] < 60,
    "UNKNOWN",
    "RELIABLE"
)

print("=== COVERAGE STATUS ===")

print(
    "Total districts:",
    len(district_coverage)
)

print(
    "UNKNOWN districts:",
    (
        district_coverage["Coverage_Status"]
        == "UNKNOWN"
    ).sum()
)

print(
    "RELIABLE districts:",
    (
        district_coverage["Coverage_Status"]
        == "RELIABLE"
    ).sum()
)

print("\nStatus distribution:")

display(
    district_coverage["Coverage_Status"]
    .value_counts()
)

=== COVERAGE STATUS ===
Total districts: 340
UNKNOWN districts: 257
RELIABLE districts: 83

Status distribution:


Coverage_Status
UNKNOWN     257
RELIABLE     83
Name: count, dtype: int64

In [35]:
# ============================================================
# CELL 7 — UNKNOWN DISTRICT LIST
# ============================================================

unknown_districts = (
    district_coverage[
        district_coverage["Coverage_Status"] == "UNKNOWN"
    ]
    [
        [
            "District_Code",
            "District_Name",
            "Mean_Panel_Coverage_Pct",
            "Coverage_Status"
        ]
    ]
    .sort_values(
        "Mean_Panel_Coverage_Pct"
    )
    .reset_index(drop=True)
)

print("=== UNKNOWN DISTRICTS ===")

print(
    "Number of UNKNOWN districts:",
    len(unknown_districts)
)

display(unknown_districts)

=== UNKNOWN DISTRICTS ===
Number of UNKNOWN districts: 257


,District_Code,District_Name,Mean_Panel_Coverage_Pct,Coverage_Status
0,DST_0247,Sikar,10.704167,UNKNOWN
1,DST_0071,Tiruppur,11.182917,UNKNOWN
2,DST_0159,Anand,11.340417,UNKNOWN
3,DST_0209,Rangareddy,12.164583,UNKNOWN
4,DST_0076,Kancheepuram,12.455833,UNKNOWN
...,...,...,...,...
252,DST_0093,Tirupathur,59.019583,UNKNOWN
253,DST_0212,Hanumakonda,59.234583,UNKNOWN
254,DST_0037,Bengaluru Rural,59.397083,UNKNOWN
255,DST_0323,Betul,59.542083,UNKNOWN


In [36]:
# ============================================================
# CELL 8 — FINAL COVERAGE VALIDATION
# ============================================================

print("=== FINAL COVERAGE VALIDATION ===")

# Check for UNKNOWN districts incorrectly at/above 60%
unknown_invalid = district_coverage[
    (district_coverage["Coverage_Status"] == "UNKNOWN") &
    (district_coverage["Mean_Panel_Coverage_Pct"] >= 60)
]

# Check for RELIABLE districts incorrectly below 60%
reliable_invalid = district_coverage[
    (district_coverage["Coverage_Status"] == "RELIABLE") &
    (district_coverage["Mean_Panel_Coverage_Pct"] < 60)
]

# Check missing coverage
missing_coverage = district_coverage[
    "Mean_Panel_Coverage_Pct"
].isna().sum()

print(
    "UNKNOWN districts with coverage >= 60%:",
    len(unknown_invalid)
)

print(
    "RELIABLE districts with coverage < 60%:",
    len(reliable_invalid)
)

print(
    "Districts with missing coverage:",
    missing_coverage
)

if (
    len(unknown_invalid) == 0
    and len(reliable_invalid) == 0
    and missing_coverage == 0
):
    print("\n✓ COVERAGE VALIDATION PASSED")
else:
    print("\n⚠ COVERAGE VALIDATION FAILED")

=== FINAL COVERAGE VALIDATION ===
UNKNOWN districts with coverage >= 60%: 0
RELIABLE districts with coverage < 60%: 0
Districts with missing coverage: 0

✓ COVERAGE VALIDATION PASSED


In [37]:
# ============================================================
# CELL 9 — SAVE PHASE 3 COVERAGE OUTPUTS
# ============================================================

OUTPUT_DATA.mkdir(
    parents=True,
    exist_ok=True
)

# Complete district-level coverage analysis
district_coverage.to_csv(
    OUTPUT_DATA / "district_coverage_analysis.csv",
    index=False
)

# UNKNOWN districts kept separately
unknown_districts.to_csv(
    OUTPUT_DATA / "unknown_districts_low_coverage.csv",
    index=False
)

print("=== PHASE 3 OUTPUTS SAVED ===")

print(
    "District coverage:",
    OUTPUT_DATA / "district_coverage_analysis.csv"
)

print(
    "UNKNOWN districts:",
    OUTPUT_DATA / "unknown_districts_low_coverage.csv"
)

print("\nRows saved:")
print(
    "District coverage:",
    len(district_coverage)
)

print(
    "UNKNOWN districts:",
    len(unknown_districts)
)

=== PHASE 3 OUTPUTS SAVED ===
District coverage: d:\sivahitesh\Desktop\Tail_project\data\outputs\district_coverage_analysis.csv
UNKNOWN districts: d:\sivahitesh\Desktop\Tail_project\data\outputs\unknown_districts_low_coverage.csv

Rows saved:
District coverage: 340
UNKNOWN districts: 257


Phase 3 conclusion:

Panel coverage was successfully quantified across all 340 districts and 24 months. After validating the standardized join keys and complete district-month structure, 257 districts were classified as UNKNOWN because their mean panel coverage was below 60%, while 83 districts had sufficient coverage for further WCI analysis.